In [ ]:




import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from copy import deepcopy
import time

In [ ]:




base_path = r"C:\Users\mayan\Desktop\3STSEM\ai\model\Datasets"

tasks = pd.read_csv(os.path.join(base_path, "dataset_A.csv"))
edge_nodes = pd.read_csv(os.path.join(base_path, "edge_nodes.csv"))
edge_state = pd.read_csv(os.path.join(base_path, "edge_state.csv"))
cloud_nodes = pd.read_csv(os.path.join(base_path, "cloud_nodes.csv"))
cloud_state = pd.read_csv(os.path.join(base_path, "cloud_state.csv"))
network_state = pd.read_csv(os.path.join(base_path, "network_state.csv"))

print("Tasks shape:", tasks.shape)

In [ ]:

task_types = ["sensor", "image", "ai", "video"]

tasks["task_type"] = np.random.choice(
    task_types,
    size=len(tasks),
    p=[0.5, 0.25, 0.15, 0.10]
)

tasks.loc[tasks["task_type"]=="sensor", "cpu_cycles"] *= 0.5
tasks.loc[tasks["task_type"]=="image", "cpu_cycles"] *= 1.5
tasks.loc[tasks["task_type"]=="ai", "cpu_cycles"] *= 3
tasks.loc[tasks["task_type"]=="video", "cpu_cycles"] *= 2
network_state["load_factor"] = 1 + 0.5 * np.sin(
    2 * np.pi * network_state["timestep"] / 200
)

network_state["network_delay_ms"] *= network_state["load_factor"]
for idx, row in network_state.iterrows():
    if row["network_delay_ms"] > 70:
        edge_state.loc[
            edge_state["timestep"] == row["timestep"],
            "edge_queue_length"
        ] += np.random.randint(5,15)

In [ ]:




def compute_latency(task_row):

    t = task_row["arrival_time"]
    edge_id = task_row["assigned_edge_id"]

    edge = edge_state[
        (edge_state["timestep"] == t) &
        (edge_state["edge_id"] == edge_id)
    ].iloc[0]

    net = network_state[
        network_state["timestep"] == t
    ].iloc[0]

    cloud_t = cloud_state[
        cloud_state["timestep"] == t
    ]

    cloud_cpu = cloud_t["cloud_cpu_available"].mean()
    cloud_queue = cloud_t["cloud_queue_length"].mean()


    edge_latency = (
        task_row["cpu_cycles"] / edge["edge_cpu_available"]
        + edge["edge_queue_length"] * 0.5
    )


    cloud_latency = (
        task_row["task_size_mb"] / net["uplink_bandwidth"]
        + task_row["cpu_cycles"] / cloud_cpu
        + net["network_delay_ms"]
        + cloud_queue * 0.5
    )

    return 0 if edge_latency < cloud_latency else 1


In [ ]:




print("Generating offloading labels...")
tasks["offload_label"] = tasks.apply(compute_latency, axis=1)

print(tasks["offload_label"].value_counts())

In [ ]:




def build_features(task_row):

    t = task_row["arrival_time"]
    edge_id = task_row["assigned_edge_id"]

    edge = edge_state[
        (edge_state["timestep"] == t) &
        (edge_state["edge_id"] == edge_id)
    ].iloc[0]

    net = network_state[
        network_state["timestep"] == t
    ].iloc[0]

    cloud_t = cloud_state[
        cloud_state["timestep"] == t
    ]

    return [
        task_row["task_size_mb"],
        task_row["cpu_cycles"],
        task_row["memory_req_mb"],
        task_row["deadline_ms"],
        task_row["priority_level"],
        task_row["security_sensitivity"],
        edge["edge_cpu_available"],
        edge["edge_memory_available"],
        edge["edge_queue_length"],
        net["network_delay_ms"],
        net["uplink_bandwidth"],
        cloud_t["cloud_cpu_available"].mean(),
        cloud_t["cloud_queue_length"].mean()
    ]

features = np.array(tasks.apply(build_features, axis=1).tolist())
labels = tasks["offload_label"].values

print("Feature shape:", features.shape)

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
scaler = StandardScaler()
features = scaler.fit_transform(features)

In [ ]:





cutoff_time = 700


train_mask = tasks["arrival_time"] <= cutoff_time
test_mask  = tasks["arrival_time"] > cutoff_time

X_train = features[train_mask]
X_test  = features[test_mask]

y_train = labels[train_mask]
y_test  = labels[test_mask]

print("Train samples:", len(X_train))
print("Test samples :", len(X_test))




scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.squeeze(), dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.squeeze(), dtype=torch.long)

In [ ]:




class OffloadNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.layers(x)

model = OffloadNet(X_train.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:




def split_clients(X, y, num_clients=5):
    split_size = len(X) // num_clients
    clients = []
    for i in range(num_clients):
        start = i * split_size
        end = (i + 1) * split_size
        clients.append((X[start:end], y[start:end]))
    return clients

clients = split_clients(X_train, y_train, num_clients=5)

In [ ]:




def federated_average(models):
    global_model = deepcopy(models[0])
    for key in global_model.state_dict().keys():
        global_model.state_dict()[key] = torch.mean(
            torch.stack([m.state_dict()[key].float() for m in models]), dim=0
        )
    return global_model

rounds = 20

for r in range(rounds):
    local_models = []

    for client_X, client_y in clients:
        local_model = deepcopy(model)
        opt = optim.Adam(local_model.parameters(), lr=0.001)

        for epoch in range(2):
            opt.zero_grad()
            output = local_model(client_X)
            loss = criterion(output, client_y)
            loss.backward()
            opt.step()

        local_models.append(local_model)

    model = federated_average(local_models)
    print(f"Federated Round {r+1} completed")

In [ ]:




with torch.no_grad():
    start = time.time()
    predictions = model(X_test)
    decision_time = time.time() - start

    predicted = torch.argmax(predictions, dim=1)
    accuracy = (predicted == y_test).float().mean()

print("Test Accuracy:", accuracy.item())
print("Decision Overhead Time:", decision_time)

In [ ]:




class ResourceAllocator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.layers(x)

allocator = ResourceAllocator(X_train.shape[1])

In [ ]:




def generate_allocation_targets(task_row):

    t = task_row["arrival_time"]
    edge_id = task_row["assigned_edge_id"]

    edge = edge_state[
        (edge_state["timestep"] == t) &
        (edge_state["edge_id"] == edge_id)
    ].iloc[0]


    cpu_share = task_row["cpu_cycles"] / edge["edge_cpu_available"]
    cpu_share = np.clip(cpu_share, 0.01, 1.0)


    mem_share = task_row["memory_req_mb"] / edge["edge_memory_available"]
    mem_share = np.clip(mem_share, 0.01, 1.0)

    return [cpu_share, mem_share]

allocation_targets = np.array(tasks.apply(generate_allocation_targets, axis=1).tolist())

In [ ]:




edge_mask = tasks["offload_label"] == 0

X_alloc = features[edge_mask]
y_alloc = allocation_targets[edge_mask]

X_alloc = torch.tensor(X_alloc, dtype=torch.float32)
y_alloc = torch.tensor(y_alloc, dtype=torch.float32)

print("Edge allocation samples:", X_alloc.shape)

In [ ]:




class ResourceAllocator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.layers(x)

allocator = ResourceAllocator(X_alloc.shape[1])
criterion_alloc = nn.MSELoss()
optimizer_alloc = optim.Adam(allocator.parameters(), lr=0.001)

In [ ]:




epochs = 15

for epoch in range(epochs):
    optimizer_alloc.zero_grad()
    outputs = allocator(X_alloc)
    loss = criterion_alloc(outputs, y_alloc)
    loss.backward()
    optimizer_alloc.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:




with torch.no_grad():
    preds = allocator(X_alloc)
    error = torch.mean(torch.abs(preds - y_alloc))

print("Mean Allocation Error:", error.item())

In [ ]:



import matplotlib.pyplot as plt
federated_losses = []

for r in range(rounds):
    local_losses = []
    local_models = []

    for client_X, client_y in clients:
        local_model = deepcopy(model)
        opt = optim.Adam(local_model.parameters(), lr=0.001)

        for epoch in range(2):
            opt.zero_grad()
            output = local_model(client_X)
            loss = criterion(output, client_y)
            loss.backward()
            opt.step()

        local_losses.append(loss.item())
        local_models.append(local_model)

    model = federated_average(local_models)
    federated_losses.append(np.mean(local_losses))

plt.plot(federated_losses)
plt.title("Federated Learning Convergence")
plt.xlabel("Communication Rounds")
plt.ylabel("Average Loss")
plt.show()

In [ ]:




from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

with torch.no_grad():
    preds = model(X_test)
    pred_labels = torch.argmax(preds, dim=1).numpy()

cm = confusion_matrix(y_test.numpy(), pred_labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["Edge","Cloud"])

disp.plot()
plt.title("Stage 1: Confusion Matrix")
plt.show()

In [ ]:




from sklearn.metrics import roc_curve, auc

with torch.no_grad():
    probs = torch.softmax(model(X_test), dim=1)[:,1].numpy()

fpr, tpr, _ = roc_curve(y_test.numpy(), probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title(f"Stage 1 ROC Curve (AUC={roc_auc:.3f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test.numpy(), probs)

plt.plot(recall, precision)
plt.title("Stage 1 Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

In [ ]:




with torch.no_grad():
    alloc_preds = allocator(X_alloc)
    errors = torch.abs(alloc_preds - y_alloc).numpy()

plt.hist(errors.flatten(), bins=50)
plt.title("Stage 2: Allocation Absolute Error Distribution")
plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.show()

In [ ]:

plt.scatter(y_alloc[:,0], alloc_preds[:,0], alpha=0.3)
plt.title("Stage 2: CPU Allocation Prediction")
plt.xlabel("True CPU Share")
plt.ylabel("Predicted CPU Share")
plt.show()


plt.scatter(y_alloc[:,1], alloc_preds[:,1], alpha=0.3)
plt.title("Stage 2: Memory Allocation Prediction")
plt.xlabel("True Memory Share")
plt.ylabel("Predicted Memory Share")
plt.show()

In [ ]:




end_to_end_latency = []

sample = tasks.sample(5000)

for _, row in sample.iterrows():

    t = row["arrival_time"]
    edge_id = row["assigned_edge_id"]

    edge = edge_state[
        (edge_state["timestep"] == t) &
        (edge_state["edge_id"] == edge_id)
    ].iloc[0]

    net = network_state[
        network_state["timestep"] == t
    ].iloc[0]

    cloud_t = cloud_state[
        cloud_state["timestep"] == t
    ]

    decision = row["offload_label"]

    if decision == 0:
        latency = row["cpu_cycles"] / edge["edge_cpu_available"]
    else:
        latency = (
            row["task_size_mb"] / net["uplink_bandwidth"]
            + row["cpu_cycles"] / cloud_t["cloud_cpu_available"].mean()
        )

    end_to_end_latency.append(latency)

plt.hist(end_to_end_latency, bins=50)
plt.title("Full Model: End-to-End Latency Distribution")
plt.xlabel("Latency")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.hist(tasks["task_size_mb"], bins=150)
plt.yscale("log")

plt.title("Realistic Heavy-Tailed IoT Workload")
plt.xlabel("Task Size (MB)")
plt.ylabel("Frequency (log scale)")

plt.show()

In [ ]:
decision_counts = tasks["offload_label"].value_counts()

plt.figure(figsize=(6,5))
plt.bar(["Edge Execution", "Cloud Offloading"],
        decision_counts.values)

plt.title("Hierarchical Offloading Decisions")
plt.ylabel("Number of Tasks")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.plot(federated_losses)

plt.title("Federated Training Stability")
plt.xlabel("Communication Rounds")
plt.ylabel("Loss")

plt.show()